# ECO375 — FinBERT sentiment for news headlines

Scores every distinct filtered headline as **positive / negative / neutral** with FinBERT (free, open-source; no API, no daily limit).

**Before you run**
1. **Runtime → Change runtime type → T4 GPU** (free). CPU also works, just much slower.
2. Put `finbert_input_headlines.csv.gz` (from `Panel Build/data/_10_work/` in the project folder) in your Google Drive folder **`MyDrive/ECO375_sentiment/`**. If it isn't there, the notebook asks you to upload it.
3. **Runtime → Run all.** Allow Google Drive access when asked.

**Progress is saved to Drive** after every 50,000 headlines. If Colab disconnects, just **Run all** again: finished chunks are skipped.

**Output:** `MyDrive/ECO375_sentiment/finbert_labels.csv.gz` (one row per headline: `headline_id, label, p_positive, p_negative, p_neutral`) and `finbert_run_info.txt`. Put both in `Panel Build/data/_10_work/`, then run `10b_merge_sentiment.py`.

In [ ]:
# ---- settings ----
MODEL_NAME  = "ProsusAI/finbert"      # alternative: "yiyanghkust/finbert-tone" (Huang, Wang & Yang 2023)
DRIVE_DIR   = "/content/drive/MyDrive/ECO375_sentiment"
INPUT_FILE  = "finbert_input_headlines.csv.gz"
OUTPUT_FILE = "finbert_labels.csv.gz"
USE_DRIVE   = True     # False = work in /content only (progress is lost if Colab disconnects)
CHUNK       = 50_000   # headlines per saved chunk
BATCH       = 256      # headlines per GPU batch (lower to 64 if you get an out-of-memory error)
MAX_LEN     = 128      # tokens; headlines are ~15 tokens at the median, ~41 at the 99th percentile

In [ ]:
import os, glob, time, shutil, datetime
import numpy as np, pandas as pd, torch

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = DRIVE_DIR
else:
    WORK = "/content/ECO375_sentiment"
os.makedirs(os.path.join(WORK, "parts"), exist_ok=True)

inp = os.path.join(WORK, INPUT_FILE)
if not os.path.exists(inp):
    from google.colab import files
    print(f"{INPUT_FILE} not found in {WORK}: choose it in the upload box below")
    up = files.upload()
    shutil.move(next(iter(up)), inp)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: switch the runtime to T4 GPU for speed)")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers

df = pd.read_csv(inp, keep_default_na=False)
df["title"] = df["title"].astype(str)
print(f"headlines to score: {len(df):,}")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
if device == "cuda":
    model.half()
LABELS = {int(i): l.lower() for i, l in model.config.id2label.items()}
assert sorted(LABELS.values()) == ["negative", "neutral", "positive"], LABELS
print("model labels:", LABELS)

In [ ]:
def score(texts):
    # sort by length so each batch has similar lengths (less padding, faster); results go back in input order
    order = np.argsort([len(t) for t in texts], kind="stable")
    probs = np.zeros((len(texts), len(LABELS)), dtype=np.float32)
    n_trunc = 0
    with torch.inference_mode():
        for s in range(0, len(texts), BATCH):
            idx = order[s:s + BATCH]
            enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
            n_trunc += int((enc["attention_mask"].sum(1) >= MAX_LEN).sum())
            enc = {k: v.to(device) for k, v in enc.items()}
            probs[idx] = torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy()
    return probs, n_trunc

n_chunks = (len(df) + CHUNK - 1) // CHUNK
t0, done_now = time.time(), 0
for c in range(n_chunks):
    part = os.path.join(WORK, "parts", f"part_{c:03d}.csv.gz")
    if os.path.exists(part):
        continue                                   # finished in an earlier run
    sub = df.iloc[c * CHUNK:(c + 1) * CHUNK]
    probs, n_trunc = score(sub["title"].tolist())
    out = pd.DataFrame({"headline_id": sub["headline_id"].to_numpy()})
    out["label"] = [LABELS[i] for i in probs.argmax(1)]
    for i, l in LABELS.items():
        out[f"p_{l}"] = probs[:, i].round(4)
    out.to_csv(part + ".tmp", index=False, compression="gzip")
    os.replace(part + ".tmp", part)                # a chunk counts as done only once fully written
    done_now += len(sub)
    rate = done_now / (time.time() - t0)
    left = (len(df) - (c + 1) * CHUNK) / rate if rate else 0
    print(f"chunk {c + 1}/{n_chunks} saved | {rate:,.0f} headlines/s | truncated {n_trunc} | ~{max(left, 0) / 60:.1f} min left")
print("all chunks done")

In [ ]:
parts = sorted(glob.glob(os.path.join(WORK, "parts", "part_*.csv.gz")))
assert len(parts) == n_chunks, f"{len(parts)} of {n_chunks} chunks done: run the previous cell again"
res = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
assert len(res) == len(df) and res.headline_id.is_unique and set(res.headline_id) == set(df.headline_id)
res = res[["headline_id", "label", "p_positive", "p_negative", "p_neutral"]].sort_values("headline_id")
res.to_csv(os.path.join(WORK, OUTPUT_FILE), index=False)

with open(os.path.join(WORK, "finbert_run_info.txt"), "w") as f:
    f.write(f"model: {MODEL_NAME}\ntransformers: {transformers.__version__}\ntorch: {torch.__version__}\n"
            f"device: {device}\nmax_len: {MAX_LEN}\nheadlines: {len(res)}\n"
            f"finished: {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M} UTC\n")

m = df[["headline_id", "title", "n_rows"]].merge(res, on="headline_id")
print("share of distinct headlines:", m.label.value_counts(normalize=True).round(3).to_dict())
print("share of ticker-article rows:", (m.groupby("label").n_rows.sum() / m.n_rows.sum()).round(3).to_dict())
for l in ["positive", "negative"]:
    print(f"\nmost confident {l}:")
    for t in m.nlargest(5, f"p_{l}").title:
        print("  ", t[:120])

In [ ]:
# download the results (they are also in your Google Drive folder)
from google.colab import files
files.download(os.path.join(WORK, OUTPUT_FILE))
files.download(os.path.join(WORK, "finbert_run_info.txt"))